# How Playground S6E9 works

**Predicting Electric Vehicle Purchases** is a tabular binary-classification contest. You submit a CSV of **probabilities**, not Yes/No labels.

Run this notebook with the workspace venv:

1. Command palette → **Notebook: Select Notebook Kernel**
2. Pick **Python (analytics-engineering)** (or the kernel from `/home/gsdeepakkumar/analytics-engineering/.venv`)

Each cell is short. Read the markdown, run the cell, look at the printout, then go on.

## 1. What you submit and how you win

Kaggle scores `submission.csv` with **ROC AUC** between your predicted **P(Will_Buy_EV = Yes)** and the hidden test labels.

| You send | Meaning |
|---|---|
| `id` | test row id (must match `sample_submission.csv`) |
| `Will_Buy_EV` | probability in `[0, 1]`, not `Yes`/`No` |

The public leaderboard is a slice of test. The private leaderboard (final rank) is the rest. **Higher AUC wins.** Hard 0/1 predictions are almost always worse than well-calibrated probabilities.

Deadline: **2026-09-30**. Daily cap: **10** submissions.

In [ ]:
from pathlib import Path

import pandas as pd

def find_data() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here.parent.parent]:
        p = cand / "data" / "train.csv"
        if p.exists():
            return p.parent
        p = cand / "playground-series-s6e9" / "data" / "train.csv"
        if p.exists():
            return p.parent
    raise FileNotFoundError("Could not find data/train.csv. Download with kaggle competitions download playground-series-s6e9 -p playground-series-s6e9/data")

DATA = find_data()
print("data dir:", DATA)
print("files:", sorted(p.name for p in DATA.iterdir()))

## 2. A real training row

`train.csv` has ~669k people. Target `Will_Buy_EV` is `Yes`/`No` (about **17.5% Yes**). `test.csv` has the same columns except the target. No missing values.

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

print("train", train.shape, "test", test.shape)
print("columns", list(train.columns))
print("Yes rate", (train["Will_Buy_EV"] == "Yes").mean())
print("\n--- one observation ---")
print(train.iloc[2].to_string())

## 3. Legal prediction format

Sample submission fills every row with the **global Yes rate** (~0.1746). That is a valid file and scores **AUC = 0.5** (no ranking skill).

In [ ]:
print(sample.head())
print("unique values in sample:", sample["Will_Buy_EV"].unique())

# Tiny legal example: same ids, dummy probabilities that still rank some rows higher.
tiny = sample.head(5).copy()
tiny["Will_Buy_EV"] = [0.12, 0.40, 0.81, 0.05, 0.33]
print("\nlegal 5-row example:")
print(tiny)

## 4. Scoring, leakage, and generator quirks

AUC only cares about **rank order**: people you scored higher should be more likely to be Yes.

Gotchas:

- **Do not train on `test.csv` labels** — there are none. Using test *features* for encodings is common; stuffing test target is impossible here.
- **`id` is just a row index** (train 0…668664, test 668665…). Do not treat it as a feature.
- Data is **synthetic** (Playground). Public discussion recovered an approximate generator: income, environmental concern, subsidy, and range anxiety dominate. Trees still beat a raw formula because of extra noise and extra columns.
- Hard edges exist (e.g. very high income → always Yes in train). Do not overfit those cliffs with one-off if-rules unless CV agrees.
- Strong public kernels use **gradient boosting** (LightGBM / XGBoost). Extra tricks people talk about (digit splits of numbers, original 10k dataset) are **later versions**, not required for a first legal submit.

In [ ]:
from sklearn.metrics import roc_auc_score

y = (train["Will_Buy_EV"] == "Yes").astype(int)
constant = y.mean()
print("constant-prob AUC (should be 0.5):", roc_auc_score(y, [constant] * len(y)))

# Income ranks buyers somewhat — a one-feature sanity check, not a model.
print("income-as-score AUC:", roc_auc_score(y, train["Annual_Income_USD"]))
print("subsidy Yes rate:", y[train["Subsidy_Available"] == "Yes"].mean())
print("subsidy No rate:", y[train["Subsidy_Available"] == "No"].mean())

## 5. Local eval and submit

From the repo root, with `.venv` active:

```bash
python playground-series-s6e9/scripts/eval_local.py
python playground-series-s6e9/main.py   # writes playground-series-s6e9/submission.csv
```

`eval_local.py` is 5-fold **out-of-fold** AUC on train (each row predicted by a model that did not see it). Expect ~**0.941–0.942** for v1 raw XGBoost. Public LB leaders sit near **0.9467** — a small gap in AUC points, a large gap in rank.

Submit only after we agree in chat (do not paste tokens):

```bash
kaggle competitions submit playground-series-s6e9 -f playground-series-s6e9/submission.csv -m "v1 xgb raw features"
kaggle competitions submissions playground-series-s6e9
```